# European benchmark: GTFS reference-day metrics
**IE Sustainability Datathon (Kearney) · Deliverable 1 · Supporting notebook**

Computes `n_bus_lines` and `daily_bus_km` for each comparison city on a disclosed reference day, filtered to the brief's mandatory network perimeter (section 4.4). Helsinki is the fully reproducible peer. Output: `data_derived/benchmark_gtfs_intermediate.csv`, read by the main notebook, which writes `File_1.csv`. This notebook does not write any mandatory CSV.

In [ ]:
# Environment: runs as-is in Google Colab (Python 3.10+).
!pip -q install pandas numpy pyproj shapely folium plotly kaleido requests

import os, io, json, zipfile, datetime as dt, hashlib, textwrap
import numpy as np, pandas as pd, requests

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

BASE = "/content/datathon" if os.path.exists("/content") else os.path.abspath("..")
RAW, DERIVED, OUT = (os.path.join(BASE, d) for d in ("data_raw", "data_derived", "outputs"))
for d in (RAW, DERIVED, OUT):
    os.makedirs(d, exist_ok=True)

RUN_DATE = dt.date.today().isoformat()
import sys, platform
print(f"Run date {RUN_DATE} | Python {platform.python_version()} | pandas {pd.__version__} | numpy {np.__version__}")
print("Folders:", RAW, DERIVED, OUT, sep="\n  ")

In [ ]:
# ---- Data register: every downloaded or manually obtained source is logged here.
DATA_REGISTER = []

def fetch(url, filename, note="", manual=False):
    """Download a public file into data_raw/ and log source, date, size and checksum.
    manual=True: the file was obtained by hand (registration/API); it must already be in data_raw/."""
    path = os.path.join(RAW, filename)
    if not manual and not os.path.exists(path):
        r = requests.get(url, timeout=120)
        r.raise_for_status()
        with open(path, "wb") as f:
            f.write(r.content)
    size = os.path.getsize(path) if os.path.exists(path) else None
    sha = hashlib.sha256(open(path, "rb").read()).hexdigest()[:12] if size else None
    DATA_REGISTER.append(dict(file=filename, url=url, downloaded=RUN_DATE, bytes=size,
                              sha256_12=sha, obtained="manual" if manual else "script", note=note))
    return path

def show_register():
    display(pd.DataFrame(DATA_REGISTER))

# ---- Provenance labels (brief section 4.6): V verified, D derived, P proxy, A assumption.
ASSUMPTIONS = []

def assume(variable, value, label, evidence, impact="", sensitivity=""):
    assert label in {"V", "D", "P", "A"}, "label must be V, D, P or A"
    ASSUMPTIONS.append(dict(variable=variable, value=value, label=label, evidence=evidence,
                            impact=impact, sensitivity=sensitivity))
    return value

# ---- Visible checks: every step prints CHECK ... PASS / FAIL.
CHECKS = []

def check(name, ok, detail=""):
    CHECKS.append(dict(check=name, result="PASS" if ok else "FAIL", detail=detail))
    print(f"CHECK {'PASS' if ok else 'FAIL'} | {name}" + (f" | {detail}" if detail else ""))
    return ok

In [ ]:
# Fixed parameters from the official brief, section 4.5. Do not hard-code these numbers anywhere else.
P = dict(
    kwh_km      = {"Standard": 1.30, "Articulated": 1.85},   # battery energy incl. HVAC
    range_km    = {"Standard": 300,  "Articulated": 240},    # effective daily range (75% of tender minimum)
    cng_kg_km   = {"Standard": 0.45, "Articulated": 0.65},
    charger_kw  = 100,
    window_h    = 7,                                          # 23:00-06:00
    charge_eff  = 0.90,
    bus_price_eur = {"Standard": 615_000, "Articulated": 800_000},  # excl. VAT
    benchmark_kwh_km = 1.30,
    service_days = 365,
    # TODO(owner: emissions) confirm value, unit and boundary against the cited page before submission
    cng_kgco2_per_kg = assume("CO2 factor CNG", 2.7175, "V", "MITECO emission factors (inventory annex VII)", "Files 2, 4"),
    grid_gco2_kwh    = assume("CO2 factor electricity Spain 2025", 108.4, "V", "Red Electrica 2025 generation emissions", "Files 1, 2, 4"),
)
display(pd.DataFrame([(k, v) for k, v in P.items()], columns=["parameter", "value"]))

In [ ]:
# ---- Generic GTFS helpers (shared logic with the benchmark notebook).
def read_gtfs(zip_path, tables=("agency","routes","trips","stop_times","calendar","calendar_dates","shapes","feed_info")):
    feed = {}
    with zipfile.ZipFile(zip_path) as z:
        names = set(z.namelist())
        for t in tables:
            if f"{t}.txt" in names:
                feed[t] = pd.read_csv(z.open(f"{t}.txt"), dtype=str, low_memory=False)
    return feed

def active_service_ids(feed, date_str):
    """Service ids running on date_str (YYYY-MM-DD), applying calendar then calendar_dates exceptions."""
    d = pd.Timestamp(date_str); ymd = d.strftime("%Y%m%d"); dow = d.day_name().lower()
    active = set()
    if "calendar" in feed:
        c = feed["calendar"]
        m = (c["start_date"] <= ymd) & (c["end_date"] >= ymd) & (c[dow] == "1")
        active |= set(c.loc[m, "service_id"])
    if "calendar_dates" in feed:
        cd = feed["calendar_dates"]; cd = cd[cd["date"] == ymd]
        active |= set(cd.loc[cd["exception_type"] == "1", "service_id"])
        active -= set(cd.loc[cd["exception_type"] == "2", "service_id"])
    return active

def haversine_km(lat1, lon1, lat2, lon2):
    r = 6371.0088
    p1, p2 = np.radians(lat1), np.radians(lat2)
    dphi, dl = p2 - p1, np.radians(lon2 - lon1)
    a = np.sin(dphi/2)**2 + np.cos(p1)*np.cos(p2)*np.sin(dl/2)**2
    return 2*r*np.arcsin(np.sqrt(a))

def shape_lengths_km(feed):
    s = feed["shapes"].copy()
    for c in ("shape_pt_lat","shape_pt_lon"): s[c] = s[c].astype(float)
    s["shape_pt_sequence"] = s["shape_pt_sequence"].astype(int)
    s = s.sort_values(["shape_id","shape_pt_sequence"])
    g = s.groupby("shape_id")
    s["seg"] = haversine_km(g["shape_pt_lat"].shift(), g["shape_pt_lon"].shift(), s["shape_pt_lat"], s["shape_pt_lon"])
    return s.groupby("shape_id")["seg"].sum()

def trip_km(feed, trip_ids, dist_unit_m=None):
    """Km per trip. Preferred: shape_dist_traveled (unit must be confirmed: metres or km);
    fallback: geodesic length of the trip's shape. Prints which method each trip used."""
    st = feed["stop_times"][feed["stop_times"]["trip_id"].isin(trip_ids)]
    km = pd.Series(dtype=float)
    if "shape_dist_traveled" in st and st["shape_dist_traveled"].notna().any():
        d = st.assign(x=pd.to_numeric(st["shape_dist_traveled"], errors="coerce")).groupby("trip_id")["x"].agg(lambda v: v.max() - v.min())
        assert dist_unit_m is not None, "Confirm shape_dist_traveled unit (metres or km) and pass dist_unit_m=True/False"
        km = d / 1000 if dist_unit_m else d
    missing = set(trip_ids) - set(km.dropna().index)
    if missing and "shapes" in feed:
        lens = shape_lengths_km(feed)
        t = feed["trips"].set_index("trip_id").loc[list(missing), "shape_id"]
        km = pd.concat([km.dropna(), t.map(lens)])
    print(f"trip_km: {len(km)} trips, {len(missing)} via shape geometry fallback")
    return km

## Cities, feeds and perimeters
Feeds are those linked in the brief, section 4.4. Perimeter filters must be documented per city (operator / agency, route_type 3 = bus, exclusions).

In [ ]:
CITIES = {
    # city: dict(feed landing page, operator perimeter, reference date, filter notes)
    "Helsinki": dict(landing="https://www.hsl.fi/en/hsl/open-data", operator="HSL", ref_date="YYYY-MM-DD", filter="TODO: HSL bus routes, route_type 3 (and HSL extended bus types if used)"),
    "Berlin":   dict(landing="https://unternehmen.vbb.de/en/digital-services/datasets/", operator="BVG", ref_date="YYYY-MM-DD", filter="TODO: VBB feed, agency = BVG, bus routes only"),
    "Paris":    dict(landing="https://data.iledefrance-mobilites.fr/explore/dataset/offre-horaires-tc-gtfs-idfm/", operator="RATP (IDFM network)", ref_date="YYYY-MM-DD", filter="TODO: perimeter per brief 4.4; large feed, read stop_times in chunks"),
    "Brussels": dict(landing="https://data.belgianmobility.io/en/data.html?agency=stibmivb", operator="STIB-MIVB", ref_date="YYYY-MM-DD", filter="TODO: STIB bus routes; feed is a March 2026 archive — check validity"),
}
display(pd.DataFrame(CITIES).T)

In [ ]:
def city_metrics(city, zip_path, route_filter, dist_unit_m=None):
    """route_filter(feed) -> set of route_ids in the perimeter. Returns n_bus_lines and daily_bus_km on the city's reference day."""
    feed = read_gtfs(zip_path)
    ref = CITIES[city]["ref_date"]
    services = active_service_ids(feed, ref)
    routes = route_filter(feed)
    trips = feed["trips"][feed["trips"].service_id.isin(services) & feed["trips"].route_id.isin(routes)]
    km = trip_km(feed, set(trips.trip_id), dist_unit_m)
    lines = trips.merge(feed["routes"][["route_id", "route_short_name"]], on="route_id").route_short_name.nunique()
    out = dict(city=city, reference_date=ref, n_bus_lines=int(lines), daily_bus_km=round(float(km.sum()), 1), n_trips=len(trips))
    print(out)
    return out

## Helsinki (fully reproducible peer)
HSL feed (the whole Helsinki region: Helsinki, Espoo, Vantaa, Kauniainen and neighbouring municipalities). Same method as Paris, so the two rows are comparable:

- **Same reference day as Madrid** (2026-11-17, §2 of the main notebook). The snapshot is valid 10 Nov 2026 – 8 Jan 2027, so the day is inside it; the file is pinned by date and checksum in the data register.
- **Bus = HSL's extended route types 700–799.** HSL never uses `route_type` 3: its buses are 701 (regular), 702 (trunk) and 704 (local). Tram (0), metro (1), commuter rail (109), ferry (4) and light rail (900) are excluded.
- **Geographic perimeter, as for Paris:** lines serving the City of Helsinki, counting only the km run inside the city. HSL is a regional authority like IDFM, so the operator alone cannot define the city network. A stop is in Helsinki when its HSL stop code starts with `H` (`E` Espoo, `V` Vantaa, …), so no external boundary file is needed.
- **No `frequencies.txt`** in this feed: every trip is an individual run, as in Madrid's feed.

In [1]:
# Helsinki feed. HSL rebuilds it regularly, so fetch() pins the snapshot (date + sha256);
# the numbers below are reproducible only from that exact file.
HSL_GTFS = "https://infopalvelut.storage.hsldev.com/gtfs/hsl.zip"
hel_zip = fetch(HSL_GTFS, "hsl_gtfs.zip",
                "HSL GTFS, whole HSL region, snapshot valid 2026-11-10 to 2027-01-08 — "
                "https://www.hsl.fi/en/hsl/open-data")

HEL_REF = assume("Helsinki reference day", "2026-11-17", "A",
                 "Same day as Madrid's reference day (section 2), inside this feed's validity "
                 "2026-11-10 to 2027-01-08; no calendar exceptions on that day; re-check if section 2 moves",
                 "File 1 Helsinki row")
CITIES["Helsinki"]["ref_date"] = HEL_REF      # runtime only: the shared CITIES cell is left untouched

HEL_BUS_TYPES = assume("Helsinki bus route types", "700-799 (HSL uses 701, 702, 704)", "V",
                       "GTFS extended route types; HSL routes.txt has no route_type 3", "File 1 Helsinki row")
HEL_STOP_PREFIX = assume("Helsinki city stops", "stop_code starting with H + digit", "P",
                         "HSL stop codes carry the municipality letter (H Helsinki, E Espoo, V Vantaa, ...); "
                         "Hy (Hyvinkaa) is excluded by requiring a digit after H", "File 1 Helsinki row")
print("feed:", os.path.basename(hel_zip), "| reference day:", HEL_REF)

feed: hsl_gtfs.zip | reference day: 2026-11-17


**Read and clean.** Same steps as the Madrid data preparation (notebook 03): read everything as text, strip padded text, drop exact duplicate rows, then select the bus trips active on the reference day. `stop_times.txt` (1.1 GB) is not loaded here; it is streamed in the next cell.

In [2]:
hel_feed = read_gtfs(hel_zip, tables=("agency", "routes", "trips", "calendar", "calendar_dates", "stops", "shapes", "feed_info"))
_clean = []
for t, df in hel_feed.items():
    obj = df                                   # read_gtfs reads every column as text
    padded = int((obj.notna() & (obj != obj.apply(lambda c: c.str.strip()))).sum().sum())
    df = df.apply(lambda c: c.str.strip()); dups = int(df.duplicated().sum())
    hel_feed[t] = df.drop_duplicates()
    _clean.append(dict(table=t, rows=len(df), padded_cells=padded, exact_duplicates=dups))
display(pd.DataFrame(_clean))

hel_services = active_service_ids(hel_feed, HEL_REF)
hel_bus = hel_feed["routes"][hel_feed["routes"].route_type.str.fullmatch(r"7\d\d")]
ht = hel_feed["trips"]
ht = ht[ht.service_id.isin(hel_services) & ht.route_id.isin(set(hel_bus.route_id))]
_stops = hel_feed["stops"]
hel_city_stops = set(_stops.loc[_stops.stop_code.str.match(r"^H\d", na=False), "stop_id"])

_fi = hel_feed["feed_info"].iloc[0]
check("Helsinki: reference day inside the feed validity",
      _fi.feed_start_date <= HEL_REF.replace("-", "") <= _fi.feed_end_date,
      f"feed valid {_fi.feed_start_date} to {_fi.feed_end_date}")
check("Helsinki: no exact duplicate trips", not ht.duplicated().any() and ht.trip_id.is_unique)
check("Helsinki: every active bus trip has a shape", bool(ht.shape_id.notna().all()))
check("Helsinki: feed has no frequencies.txt (trips are individual runs)",
      "frequencies.txt" not in zipfile.ZipFile(hel_zip).namelist())
print(f"{len(ht)} active bus trips | {ht.route_id.nunique()} bus routes | "
      f"route types {sorted(hel_bus.route_type.unique())} | {len(hel_city_stops)} Helsinki city stops")

,table,rows,padded_cells,exact_duplicates
0,agency,1,0,0
1,routes,459,0,0
2,trips,421067,0,0
3,calendar,7908,0,0
4,calendar_dates,3765,0,0
5,stops,8405,16307,0
6,shapes,492044,0,0
7,feed_info,1,0,0


CHECK PASS | Helsinki: reference day inside the feed validity | feed valid 20261110 to 20270108
CHECK PASS | Helsinki: no exact duplicate trips
CHECK PASS | Helsinki: every active bus trip has a shape
CHECK PASS | Helsinki: feed has no frequencies.txt (trips are individual runs)
19286 active bus trips | 390 bus routes | route types ['701', '702', '704'] | 2651 Helsinki city stops


**Km per trip.** This feed has `shape_dist_traveled` in **km** on every stop time, so trip length = last − first value (the main notebook's method A for Madrid). A stop-to-stop segment counts as *inside Helsinki* when both stops are Helsinki stops, the same both-ends-inside rule as Paris. Segments that cross the city boundary are kept apart and shown as a sensitivity. `stop_times.txt` is streamed in chunks and only the active bus trips are kept. The shape geometry gives an independent cross-check.

In [3]:
_ids, _parts = set(ht.trip_id), []
with zipfile.ZipFile(hel_zip) as z:
    for ch in pd.read_csv(z.open("stop_times.txt"), chunksize=3_000_000, dtype=str,
                          usecols=["trip_id", "stop_id", "stop_sequence", "shape_dist_traveled"]):
        ch = ch[ch.trip_id.isin(_ids)]
        if len(ch):
            _parts.append(ch)
hst = pd.concat(_parts)
hst = hst.assign(seq=hst.stop_sequence.astype(int), d=hst.shape_dist_traveled.astype(float),
                 h=hst.stop_id.str.strip().isin(hel_city_stops)).sort_values(["trip_id", "seq"])
_g = hst.groupby("trip_id")
hst["seg"] = hst.d - _g.d.shift()
_prev_h = _g.h.shift().eq(True)
hst["both_in"] = hst.h & _prev_h
hst["crossing"] = (hst.h ^ _prev_h) & hst.seg.notna()

_per = hst.groupby("trip_id").agg(km=("d", lambda v: v.max() - v.min()), serves_helsinki=("h", "any"))
_per["km_in_helsinki"] = hst[hst.both_in].groupby("trip_id").seg.sum().reindex(_per.index).fillna(0.0)
_per["km_crossing"] = hst[hst.crossing].groupby("trip_id").seg.sum().reindex(_per.index).fillna(0.0)
_shape_km = hel_feed["shapes"].assign(x=lambda s: s.shape_dist_traveled.astype(float)).groupby("shape_id").x.max()
ht = (ht.merge(_per, left_on="trip_id", right_index=True, how="left")
        .merge(hel_bus[["route_id", "route_short_name", "route_type"]], on="route_id", how="left")
        .assign(shape_km=lambda t: t.shape_id.map(_shape_km)))

check("Helsinki: every active bus trip has stop times", bool(ht.km.notna().all()))
check("Helsinki: every trip has positive km", bool((ht.km > 0).all()))
_ratio = ht.km / ht.shape_km
check("Helsinki: stop-time km within 1% of shape length for >= 99% of trips",
      bool(((_ratio - 1).abs() < 0.01).mean() >= 0.99), f"median ratio {_ratio.median():.4f}")
check("Helsinki: km inside the city never exceed the trip's total km",
      bool((ht.km_in_helsinki <= ht.km + 1e-6).all()))
display(ht.groupby("route_type").agg(trips=("trip_id", "size"), km=("km", "sum")).round(1))

CHECK PASS | Helsinki: every active bus trip has stop times
CHECK PASS | Helsinki: every trip has positive km
CHECK PASS | Helsinki: stop-time km within 1% of shape length for >= 99% of trips | median ratio 1.0000
CHECK PASS | Helsinki: km inside the city never exceed the trip's total km


,trips,km
route_type,,
701,16101,227943.6
702,2780,57386.4
704,405,4397.2


**Perimeter and result.** As for Paris, the alternative perimeters are printed for the report but never written to `File_1`. The per-line table flags high-km lines with the 1.5 × IQR rule; they are real trunk lines and are **kept**, as in the Madrid data preparation.

In [4]:
_all = pd.Series(True, index=ht.index)
_opts = {
    "Serves Helsinki, km inside Helsinki":          (ht.serves_helsinki, ht.km_in_helsinki),
    "  + half of the boundary-crossing segments":   (ht.serves_helsinki, ht.km_in_helsinki + 0.5 * ht.km_crossing),
    "Serves Helsinki, all km of those lines":       (ht.serves_helsinki, ht.km),
    "Helsinki internal lines only (route_id 1xxx)": (ht.route_id.str.startswith("1"), ht.km),
    "Whole HSL region":                             (_all, ht.km),
}
hel_perims = pd.DataFrame([dict(perimeter=k, n_bus_lines=int(ht.loc[m, "route_short_name"].nunique()),
                                daily_bus_km=round(float(v[m].sum()), 1), n_trips=int(m.sum()))
                           for k, (m, v) in _opts.items()])
display(hel_perims)

_km = dict(zip(hel_perims.perimeter, hel_perims.daily_bus_km))
HEL_PERIMETER = assume("Helsinki perimeter", "bus lines serving the City of Helsinki, km run inside Helsinki", "P",
                       "Same rule as Paris: HSL is a regional authority, so the city network is defined geographically; "
                       "closest analogue to EMT's municipal network",
                       "File 1 Helsinki row",
                       f"{_km['  + half of the boundary-crossing segments']:,.0f} km/day with half the boundary segments; "
                       f"{_km['Serves Helsinki, all km of those lines']:,.0f} km/day with all km of the same lines")

_m = ht.serves_helsinki
hel_lines = (ht[_m].groupby("route_short_name")
             .agg(trips=("trip_id", "size"), km_in_helsinki=("km_in_helsinki", "sum"), km_total=("km", "sum")).round(1))
_q1, _q3 = hel_lines.km_in_helsinki.quantile([0.25, 0.75])
hel_lines["iqr_outlier"] = hel_lines.km_in_helsinki > _q3 + 1.5 * (_q3 - _q1)
print(f"{int(hel_lines.iqr_outlier.sum())} high-km lines flagged (kept):")
display(hel_lines[hel_lines.iqr_outlier].sort_values("km_in_helsinki", ascending=False))
_edge = hel_lines[hel_lines.km_in_helsinki == 0]
print(f"{len(_edge)} line(s) stop in Helsinki but run 0 km between two Helsinki stops: {list(_edge.index)} "
      "(counted as serving Helsinki, same rule as Paris)")

helsinki = dict(city="Helsinki", reference_date=HEL_REF,
                n_bus_lines=int(ht.loc[_m, "route_short_name"].nunique()),
                daily_bus_km=round(float(ht.loc[_m, "km_in_helsinki"].sum()), 1),
                n_trips=int(_m.sum()))
check("Helsinki: result matches the city_metrics() contract",
      set(helsinki) == {"city", "reference_date", "n_bus_lines", "daily_bus_km", "n_trips"})
check("Helsinki: line count and daily km are positive",
      helsinki["n_bus_lines"] > 0 and helsinki["daily_bus_km"] > 0)
print(helsinki)

,perimeter,n_bus_lines,daily_bus_km,n_trips
0,"Serves Helsinki, km inside Helsinki",183,131343.4,12372
1,+ half of the boundary-crossing segments,183,136436.4,12372
2,"Serves Helsinki, all km of those lines",183,185888.5,12372
3,Helsinki internal lines only (route_id 1xxx),102,94171.4,8423
4,Whole HSL region,390,289727.1,19286


13 high-km lines flagged (kept):


,trips,km_in_helsinki,km_total,iqr_outlier
route_short_name,,,,
560,230,5115.3,6442.5,True
59,216,3623.4,3623.4,True
79,192,3462.0,3462.0,True
500,219,3456.1,3456.1,True
69,136,3419.8,3419.8,True
57,208,3344.1,3344.1,True
71,234,2931.0,2931.0,True
75,146,2848.1,2848.1,True
510,221,2821.1,6305.2,True


1 line(s) stop in Helsinki but run 0 km between two Helsinki stops: ['587'] (counted as serving Helsinki, same rule as Paris)
CHECK PASS | Helsinki: result matches the city_metrics() contract
CHECK PASS | Helsinki: line count and daily km are positive
{'city': 'Helsinki', 'reference_date': '2026-11-17', 'n_bus_lines': 183, 'daily_bus_km': 131343.4, 'n_trips': 12372}


## Berlin

In [ ]:
# TODO(owner: Berlin)
# fetch VBB feed, filter BVG bus routes, run city_metrics


## Paris

In [ ]:
# TODO(owner: Paris)
# fetch IDFM feed (large), apply brief perimeter, run city_metrics or document fallback


## Brussels

In [ ]:
# TODO(owner: Brussels)
# fetch STIB feed, check validity of the March 2026 archive, run city_metrics or document fallback


## Output
One row per city; fields not computable from GTFS are left for the main notebook (fleet, targets, charging model, grid factor).

In [ ]:
# results = [helsinki, berlin, paris, brussels]
# out = pd.DataFrame(results); out.to_csv(os.path.join(DERIVED, "benchmark_gtfs_intermediate.csv"), index=False)
# display(out); show_register()